# 03 - Test Recognition

## Testing Face Recognition System

Test the face recognition accuracy before running 24/7 monitoring.

1. 🔄 Load Face Database
2. 📹 Test with Webcam
3. 🎯 Display Recognition Results
4. ⚙️ Adjust Recognition Threshold
5. 📊 Performance Benchmarking

---

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import time
import matplotlib.pyplot as plt
from src.face_detector import FaceDetector
from src.face_recognizer import FaceRecognizer
from src.database_manager import DatabaseManager
from src.camera_handler import CameraHandler
from config import *

print("Face Recognition Test")
print("="*60)

## 1. Load Database

In [ ]:
# Initialize components
db_manager = DatabaseManager(
    known_faces_dir=DATABASE_CONFIG['known_faces_dir'],
    encodings_file=DATABASE_CONFIG['encodings_file'],
    min_images_per_person=DATABASE_CONFIG['min_images_per_person'],
    max_images_per_person=DATABASE_CONFIG['max_images_per_person']
)

# Load database
database = db_manager.load_database()

if database:
    stats = db_manager.get_database_stats()
    print(f"✅ Database loaded: {stats['num_people']} people")
    for name in database.keys():
        print(f"   - {name}")
else:
    print("❌ Database is empty. Run 02_build_face_database.ipynb first")

## 2. Initialize Recognition System

In [ ]:
face_detector = FaceDetector(
    backend=DETECTION_CONFIG['backend'],
    min_face_size=DETECTION_CONFIG['min_face_size']
)

face_recognizer = FaceRecognizer(
    model_name=RECOGNITION_CONFIG['model'],
    distance_metric=RECOGNITION_CONFIG['distance_metric'],
    threshold=RECOGNITION_CONFIG['threshold']
)

print("Recognition system initialized")
print(f"Model: {RECOGNITION_CONFIG['model']}")
print(f"Threshold: {RECOGNITION_CONFIG['threshold']}")

## 3. Test Recognition with Webcam

Run live recognition test. Press 'q' to stop.

In [ ]:
def test_recognition_live(duration_seconds=30):
    """Test recognition with live webcam feed."""
    camera = CameraHandler(
        camera_source=CAMERA_CONFIG['source'],
        width=CAMERA_CONFIG['width'],
        height=CAMERA_CONFIG['height']
    )
    
    if not camera.open():
        print("Failed to open camera")
        return
    
    print(f"Testing for {duration_seconds} seconds or until 'q' pressed...")
    start_time = time.time()
    
    while (time.time() - start_time) < duration_seconds:
        ret, frame = camera.read_frame()
        if not ret:
            break
        
        # Detect faces
        faces = face_detector.detect_faces(frame)
        
        # Recognize each face
        for face_data in faces:
            x, y, w, h = face_detector.get_face_coordinates(face_data)
            face_img = face_detector.extract_face_region(frame, face_data)
            
            if face_img is not None:
                name, confidence, distance = face_recognizer.recognize_face(face_img, database)
                
                color = (0, 255, 0) if name else (0, 0, 255)
                label = name if name else "Unknown"
                
                face_detector.draw_face_box(
                    frame, x, y, w, h,
                    color=color,
                    label=label,
                    confidence=confidence
                )
        
        # Add FPS and timestamp
        frame = camera.draw_fps(frame)
        frame = camera.draw_timestamp(frame)
        
        # Display
        key = camera.display_frame(frame, "Recognition Test")
        if key == ord('q'):
            break
    
    camera.release()
    print("Test completed")

# Run test
test_recognition_live(30)

## 4. Capture and Analyze Single Frame

In [ ]:
# Capture a frame and test recognition
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    # Detect and recognize
    faces = face_detector.detect_faces(frame)
    print(f"Detected {len(faces)} face(s)\n")
    
    for idx, face_data in enumerate(faces):
        x, y, w, h = face_detector.get_face_coordinates(face_data)
        face_img = face_detector.extract_face_region(frame, face_data)
        
        if face_img is not None:
            name, confidence, distance = face_recognizer.recognize_face(face_img, database)
            
            print(f"Face {idx+1}:")
            print(f"  Name: {name if name else 'Unknown'}")
            print(f"  Confidence: {confidence*100:.1f}%")
            print(f"  Distance: {distance:.3f}")
            print()
            
            # Draw on frame
            color = (0, 255, 0) if name else (0, 0, 255)
            label = name if name else "Unknown"
            face_detector.draw_face_box(frame, x, y, w, h, color=color, label=label, confidence=confidence)
    
    # Display result
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(frame_rgb)
    plt.title("Recognition Result")
    plt.axis('off')
    plt.show()

## Summary

### ✅ Recognition Testing Complete

If recognition is working well:
- Known faces are correctly identified
- Unknown faces are marked as such
- Confidence scores are reasonable

### 📚 Next Steps

**Run the complete security system**: `04_run_security_system.ipynb`

---